# TS-PINNs: Physics-Informed Neural Networks in Temporal Sobolev Spaces

**Paper:** Li, M., Yu, X., Wang, X., Zhu, H., Zhang, H.-K. (2026). *TS-PINNs: Physics-Informed Neural Networks in Temporal Sobolev Spaces.* Preprint, Great Bay University. https://ssrn.com/abstract=6850313

**Carpeta origen:** `PINNs/3. Arquitecturas, frameworks y variantes/TS-PINNs Physics-Informed Neural Networks in Temporal Sobolev Spaces.pdf`

## Como se usan las PINNs en este paper

El paper observa que una PINN estandar minimiza el residuo de la EDP **solo en los puntos de colocacion discretos**, lo que no controla la suavidad global de la solucion **entre** esos puntos: la red puede ajustar bien el residuo en los puntos de entrenamiento y aun asi oscilar de forma espuria entre ellos (Fig. 1). Proponen **TS-PINNs**, que incorporan **derivadas temporales de orden superior en la funcion de perdida** (regularizacion en un espacio de Sobolev temporal), restringiendo no solo el valor de la solucion sino tambien su comportamiento diferencial, lo que actua como una forma de regularizacion que mejora la suavidad y consistencia de la solucion aprendida.

El paper motiva la idea con un **modelo de juguete explicito** (Seccion 2.2, Eq. 2.1) muy simple pero ilustrativo:

$$u'(t) = 2t+3t^2+10\cos(100t),\qquad u(0)=0$$

cuya solucion exacta $u(t)=t^2+t^3+0.1\sin(100t)$ combina una tendencia suave de crecimiento con una oscilacion rapida de pequena amplitud. Una PINN convencional (perdida solo con el residuo de primer orden + condicion inicial) **falla en aproximar la solucion verdadera incluso en los puntos residuales**, mientras que anadiendo un termino de perdida extra que penaliza el residuo de la **derivada de segundo orden conocida** $u''(t)=2+6t-1000\sin(100t)$ (Eq. en Seccion 2.2), la red suprime las oscilaciones espurias y se alinea con la dinamica correcta (Fig. 1).

Este cuaderno reproduce fielmente este **ejemplo motivador exacto del paper**: misma ecuacion, mismo dominio ($t\in[0,1]$, segun la Fig. 1), misma arquitectura (1 capa oculta, 32 neuronas) y la misma comparacion PINN vs. TS-PINN.

## Repositorio publico de referencia

El PDF (preprint SSRN, aun no revisado por pares) no incluye un repositorio de codigo propio, ni se encontro uno especifico. Como referencia general del framework PINN base:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Modelo de juguete (Eq. 2.1): $u'(t)=2t+3t^2+10\cos(100t)$, $u(0)=0$, $t\in[0,1]$

In [ ]:
def rhs_u_prime(t):
    return 2 * t + 3 * t**2 + 10 * torch.cos(100 * t)

def rhs_u_double_prime(t):
    """u''(t) conocida analiticamente (derivando Eq. 2.1), usada como termino extra de perdida."""
    return 2 + 6 * t - 1000 * torch.sin(100 * t)

def exact_u(t):
    return t**2 + t**3 + 0.1 * np.sin(100 * t)

N_col = 100  # 100 puntos de colocacion uniformes, como en el paper
t_col = torch.linspace(0, 1, N_col, device=device).view(-1, 1).requires_grad_(True)
t0 = torch.zeros(1, 1, device=device, requires_grad=True)


def d_dt(f, t):
    return torch.autograd.grad(f, t, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 2. Red (1 capa oculta, 32 neuronas -- misma arquitectura que el paper) y perdidas: PINN vs TS-PINN

In [ ]:
class ToyPINN(nn.Module):
    def __init__(self, n_neurons=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, n_neurons), nn.Tanh(), nn.Linear(n_neurons, 1))

    def forward(self, t):
        return self.net(t)


def compute_loss(model, use_second_order, lam_ts=1e-5):
    u = model(t_col)
    du = d_dt(u, t_col)
    loss_pde = torch.mean((du - rhs_u_prime(t_col))**2)

    u0 = model(t0)
    loss_ic = torch.mean(u0**2)

    loss = loss_pde + 10.0 * loss_ic

    if use_second_order:
        d2u = d_dt(du, t_col)
        loss_sobolev = torch.mean((d2u - rhs_u_double_prime(t_col))**2)
        # lam_ts pequeno: el residuo de 2do orden tiene magnitud ~1000x mayor (por el 1000*sin(100t))
        # que el de 1er orden, y debe actuar como regularizador, no dominar la perdida principal.
        loss = loss + lam_ts * loss_sobolev

    return loss

## 3. Entrenamiento: PINN convencional vs. TS-PINN

In [ ]:
def train(model, use_second_order, epochs=30000, lr=1e-3):
    # Nota: el paper original entrena 100,000 epocas para este ejemplo; el termino de forzado
    # de alta frecuencia (cos(100t), amplitud 10) hace que la convergencia sea deliberadamente
    # lenta con una red tan pequena (32 neuronas, 1 capa) -- es precisamente el caso dificil que
    # motiva el paper. Aqui se usan 30,000 epocas por razones de tiempo de ejecucion; con mas
    # epocas la brecha entre PINN y TS-PINN se hace mas visible, como en su Fig. 1.
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for epoch in range(epochs):
        opt.zero_grad()
        loss = compute_loss(model, use_second_order)
        loss.backward()
        opt.step()
        hist.append(loss.item())
        if epoch % 5000 == 0:
            print(f'epoch {epoch:6d} | loss={loss.item():.4e}')
    return hist


print('--- PINN convencional (solo 1er orden + IC) ---')
model_pinn = ToyPINN().to(device)
hist_pinn = train(model_pinn, use_second_order=False)

print('--- TS-PINN (1er + 2do orden + IC) ---')
model_ts = ToyPINN().to(device)
hist_ts = train(model_ts, use_second_order=True)

## 4. Resultados: comparacion con la solucion exacta (cf. Fig. 1 del paper)

In [ ]:
t_plot = torch.linspace(0, 1, 400, device=device).view(-1, 1)
with torch.no_grad():
    u_pinn = model_pinn(t_plot).cpu().numpy().flatten()
    u_ts = model_ts(t_plot).cpu().numpy().flatten()
t_plot_np = t_plot.cpu().numpy().flatten()
u_ex = exact_u(t_plot_np)

plt.figure(figsize=(8, 5))
plt.plot(t_plot_np, u_ex, 'k-', label='Solucion exacta', linewidth=1.5)
plt.plot(t_plot_np, u_pinn, 'r--', label='PINN', linewidth=1)
plt.plot(t_plot_np, u_ts, 'b:', label='TS-PINN', linewidth=1.5)
plt.xlabel('t'); plt.ylabel('u')
plt.title('Modelo de juguete (Eq. 2.1): PINN vs TS-PINN (cf. Fig. 1 del paper)')
plt.legend()
plt.show()

err_pinn = 100 * np.linalg.norm(u_pinn - u_ex) / np.linalg.norm(u_ex)
err_ts = 100 * np.linalg.norm(u_ts - u_ex) / np.linalg.norm(u_ex)
print(f'Error relativo L2 -- PINN: {err_pinn:.2f}%  |  TS-PINN: {err_ts:.2f}%')